In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve
)
from sklearn.inspection import permutation_importance

C:\Users\shrut\AppData\Roaming\Python\Python39\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\shrut\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DASHBOARD_DIR = PROCESSED_DIR / "dashboard_tables"
REPORTS_DIR = PROJECT_ROOT / "reports"

DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROCESSED_DIR / "analytics_ready_churn_data.csv"
df = pd.read_csv(DATA_PATH)

df.shape

(7043, 28)

In [3]:
target_col = "Churn"

drop_cols = []
for c in df.columns:
    if c.lower() in ["customerid", "customer_id"]:
        drop_cols.append(c)

X = df.drop(columns=[target_col] + drop_cols)
y = df[target_col].astype(int)

y.value_counts(), y.value_counts(normalize=True).round(3)

(Churn
 0    5174
 1    1869
 Name: count, dtype: int64,
 Churn
 0    0.735
 1    0.265
 Name: proportion, dtype: float64)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((5634, 26), (1409, 26))

In [5]:
numeric_features = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
categorical_features = [c for c in X_train.columns if c not in numeric_features]

len(numeric_features), len(categorical_features)

(12, 14)

In [6]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

In [7]:
log_reg = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
)

log_reg_pipe = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", log_reg),
    ]
)

log_reg_pipe.fit(X_train, y_train)

proba_lr = log_reg_pipe.predict_proba(X_test)[:, 1]
pred_lr = (proba_lr >= 0.5).astype(int)

roc_lr = roc_auc_score(y_test, proba_lr)
ap_lr = average_precision_score(y_test, proba_lr)

roc_lr, ap_lr

(0.8447389495982845, 0.6514422576752608)

In [8]:
print("Confusion matrix (LogReg threshold=0.5):")
print(confusion_matrix(y_test, pred_lr))
print()
print(classification_report(y_test, pred_lr, digits=3))

Confusion matrix (LogReg threshold=0.5):
[[754 281]
 [ 80 294]]

              precision    recall  f1-score   support

           0      0.904     0.729     0.807      1035
           1      0.511     0.786     0.620       374

    accuracy                          0.744      1409
   macro avg      0.708     0.757     0.713      1409
weighted avg      0.800     0.744     0.757      1409



In [9]:
# Get feature names after preprocessing
ohe = log_reg_pipe.named_steps["preprocess"].named_transformers_["cat"].named_steps["onehot"]
cat_ohe_names = ohe.get_feature_names_out(categorical_features).tolist()
all_feature_names = numeric_features + cat_ohe_names

coef = log_reg_pipe.named_steps["model"].coef_.ravel()
coef_df = pd.DataFrame({"feature": all_feature_names, "coef": coef})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False).reset_index(drop=True)

top_pos = coef_df.sort_values("coef", ascending=False).head(15)[["feature", "coef"]]
top_neg = coef_df.sort_values("coef", ascending=True).head(15)[["feature", "coef"]]

top_pos, top_neg

(                             feature      coef
 1        InternetService_Fiber optic  1.077861
 4                    is_new_customer  0.527335
 6                  is_month_to_month  0.425238
 7   customer_value_segment_Low Value  0.376763
 9                 tenure_group_48-60  0.303896
 10    monthly_charge_group_Very High  0.276891
 13    PaymentMethod_Electronic check  0.250270
 14                tenure_group_60-72  0.237983
 16                 Contract_One year  0.225941
 22           Contract_Month-to-month  0.152482
 24               StreamingMovies_Yes  0.128791
 26                   StreamingTV_Yes  0.115656
 27                 OnlineSecurity_No  0.114629
 28                 MultipleLines_Yes  0.110558
 33            paperless_billing_flag  0.086723,
                                     feature      coef
 0                        InternetService_No -1.131479
 2                         tenure_group_0-12 -0.784076
 3                                    tenure -0.535114
 5         

In [10]:
hgb = HistGradientBoostingClassifier(
    random_state=42,
    learning_rate=0.08,
    max_iter=400,
)

hgb_pipe = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", hgb),
    ]
)

hgb_pipe.fit(X_train, y_train)
proba_hgb = hgb_pipe.predict_proba(X_test)[:, 1]

roc_hgb = roc_auc_score(y_test, proba_hgb)
ap_hgb = average_precision_score(y_test, proba_hgb)

roc_hgb, ap_hgb

(0.8228861505076339, 0.6180190174921573)

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_dist = {
    "model__learning_rate": [0.03, 0.05, 0.08, 0.12],
    "model__max_depth": [2, 3, 4, None],
    "model__max_leaf_nodes": [15, 31, 63, 127],
    "model__min_samples_leaf": [10, 20, 40, 80],
    "model__l2_regularization": [0.0, 0.1, 0.5, 1.0],
    "model__max_iter": [250, 400, 600],
}

search = RandomizedSearchCV(
    estimator=hgb_pipe,
    param_distributions=param_dist,
    n_iter=25,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)

search.best_score_, search.best_params_

Fitting 5 folds for each of 25 candidates, totalling 125 fits


(0.8489031588450879,
 {'model__min_samples_leaf': 10,
  'model__max_leaf_nodes': 31,
  'model__max_iter': 400,
  'model__max_depth': 2,
  'model__learning_rate': 0.03,
  'model__l2_regularization': 1.0})

In [12]:
best_hgb_pipe = search.best_estimator_
proba_best_hgb = best_hgb_pipe.predict_proba(X_test)[:, 1]

roc_best_hgb = roc_auc_score(y_test, proba_best_hgb)
ap_best_hgb = average_precision_score(y_test, proba_best_hgb)

roc_best_hgb, ap_best_hgb

(0.8437210984525564, 0.6613502943655483)

In [13]:
calibrated_hgb = CalibratedClassifierCV(
    estimator=best_hgb_pipe,
    method="isotonic",
    cv=3,
)

calibrated_hgb.fit(X_train, y_train)
proba_cal = calibrated_hgb.predict_proba(X_test)[:, 1]

roc_cal = roc_auc_score(y_test, proba_cal)
ap_cal = average_precision_score(y_test, proba_cal)

roc_cal, ap_cal

(0.8447957839262187, 0.6531559105610166)

In [14]:
prec, rec, thr = precision_recall_curve(y_test, proba_cal)
f1 = (2 * prec * rec) / (prec + rec + 1e-12)

best_idx = int(np.nanargmax(f1))
best_threshold = float(thr[best_idx]) if best_idx < len(thr) else 0.5

best_threshold, float(f1[best_idx]), float(prec[best_idx]), float(rec[best_idx])

(0.31434675005907337,
 0.6457399103134144,
 0.555984555984556,
 0.7700534759358288)

In [15]:
pred_cal = (proba_cal >= best_threshold).astype(int)

print("Confusion matrix (Calibrated HGB tuned threshold):")
print(confusion_matrix(y_test, pred_cal))
print()
print(classification_report(y_test, pred_cal, digits=3))

Confusion matrix (Calibrated HGB tuned threshold):
[[805 230]
 [ 86 288]]

              precision    recall  f1-score   support

           0      0.903     0.778     0.836      1035
           1      0.556     0.770     0.646       374

    accuracy                          0.776      1409
   macro avg      0.730     0.774     0.741      1409
weighted avg      0.811     0.776     0.785      1409



In [16]:
model_compare = pd.DataFrame(
    [
        {"model": "LogisticRegression balanced", "roc_auc": roc_lr, "avg_precision": ap_lr},
        {"model": "HGB baseline", "roc_auc": roc_hgb, "avg_precision": ap_hgb},
        {"model": "HGB tuned", "roc_auc": roc_best_hgb, "avg_precision": ap_best_hgb},
        {"model": "HGB tuned calibrated", "roc_auc": roc_cal, "avg_precision": ap_cal},
    ]
).sort_values(["roc_auc", "avg_precision"], ascending=False).reset_index(drop=True)

model_compare

,model,roc_auc,avg_precision
0,HGB tuned calibrated,0.844796,0.653156
1,LogisticRegression balanced,0.844739,0.651442
2,HGB tuned,0.843721,0.661350
3,HGB baseline,0.822886,0.618019


The tuned Gradient Boosting model slightly outperformed the baseline, but Logistic Regression achieved comparable performance with higher interpretability.

In [17]:
final_model = calibrated_hgb
final_threshold = best_threshold

final_model_name = "HGB_tuned_calibrated"
final_model_name, final_threshold

('HGB_tuned_calibrated', 0.31434675005907337)

The classification threshold was tuned using the precision-recall curve to balance recall and precision under class imbalance.

In [18]:
perm = permutation_importance(
    final_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="roc_auc",
    n_jobs=-1,
)

imp = pd.DataFrame(
    {
        "feature": X_test.columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    }
).sort_values("importance_mean", ascending=False).reset_index(drop=True)

imp.head(20)

,feature,importance_mean,importance_std
0,tenure,0.054897,0.002913
1,is_month_to_month,0.045813,0.004296
2,InternetService,0.036330,0.006877
3,Contract,0.007381,0.001770
4,TotalCharges,0.003904,0.001237
5,MonthlyCharges,0.002123,0.002586
6,PaperlessBilling,0.002114,0.001605
7,OnlineSecurity,0.002111,0.000868
8,PhoneService,0.001977,0.000687
9,PaymentMethod,0.001801,0.001477


In [19]:
final_model.fit(X, y)
all_proba = final_model.predict_proba(X)[:, 1]

pred = df.copy()
pred["churn_probability"] = all_proba
pred["predicted_churn"] = (pred["churn_probability"] >= final_threshold).astype(int)

pred[["churn_probability", "predicted_churn", "Churn"]].head()

,churn_probability,predicted_churn,Churn
0,0.728304,1,0
1,0.047554,0,0
2,0.314413,1,1
3,0.057994,0,0
4,0.681370,1,1


In [20]:
def risk_band(p: float) -> str:
    if p >= 0.75:
        return "Critical"
    if p >= 0.50:
        return "High"
    if p >= 0.25:
        return "Medium"
    return "Low"

pred["risk_band"] = pred["churn_probability"].apply(risk_band)

pred["expected_revenue_at_risk"] = pred["TotalCharges"] * pred["churn_probability"]

pred["risk_band"].value_counts()

risk_band
Low         4027
Medium      1543
High        1089
Critical     384
Name: count, dtype: int64

In [21]:
predictions_path = DASHBOARD_DIR / "customer_predictions.csv"
pred.to_csv(predictions_path, index=False)
predictions_path

WindowsPath('C:/Users/shrut/OneDrive/Desktop/customer-churn-platform/data/processed/dashboard_tables/customer_predictions.csv')

In [22]:
pred_kpis = pd.DataFrame(
    {
        "model_name": [final_model_name],
        "threshold": [round(final_threshold, 4)],
        "actual_churn_rate_pct": [round(pred["Churn"].mean() * 100, 2)],
        "predicted_churn_rate_pct": [round(pred["predicted_churn"].mean() * 100, 2)],
        "avg_churn_probability_pct": [round(pred["churn_probability"].mean() * 100, 2)],
        "total_expected_revenue_at_risk": [round(pred["expected_revenue_at_risk"].sum(), 2)],
    }
)

pred_kpis_path = DASHBOARD_DIR / "pred_kpis.csv"
pred_kpis.to_csv(pred_kpis_path, index=False)
pred_kpis_path, pred_kpis

(WindowsPath('C:/Users/shrut/OneDrive/Desktop/customer-churn-platform/data/processed/dashboard_tables/pred_kpis.csv'),
              model_name  threshold  actual_churn_rate_pct  \
 0  HGB_tuned_calibrated     0.3143                  26.54   
 
    predicted_churn_rate_pct  avg_churn_probability_pct  \
 0                     34.69                       26.6   
 
    total_expected_revenue_at_risk  
 0                      2864485.68  )

In [23]:
risk_band_summary = (
    pred.groupby("risk_band")
    .agg(
        customers=("risk_band", "count"),
        avg_churn_probability=("churn_probability", "mean"),
        predicted_churn_rate=("predicted_churn", "mean"),
        actual_churn_rate=("Churn", "mean"),
        expected_revenue_at_risk=("expected_revenue_at_risk", "sum"),
    )
    .reset_index()
)

risk_band_summary["avg_churn_probability"] = (risk_band_summary["avg_churn_probability"] * 100).round(2)
risk_band_summary["predicted_churn_rate"] = (risk_band_summary["predicted_churn_rate"] * 100).round(2)
risk_band_summary["actual_churn_rate"] = (risk_band_summary["actual_churn_rate"] * 100).round(2)
risk_band_summary["expected_revenue_at_risk"] = risk_band_summary["expected_revenue_at_risk"].round(2)

risk_band_summary_path = DASHBOARD_DIR / "risk_band_summary.csv"
risk_band_summary.to_csv(risk_band_summary_path, index=False)

risk_band_summary_path, risk_band_summary

(WindowsPath('C:/Users/shrut/OneDrive/Desktop/customer-churn-platform/data/processed/dashboard_tables/risk_band_summary.csv'),
   risk_band  customers  avg_churn_probability  predicted_churn_rate  \
 0  Critical        384                  84.73                100.00   
 1      High       1089                  60.92                100.00   
 2       Low       4027                   8.33                  0.00   
 3    Medium       1543                  35.61                 62.86   
 
    actual_churn_rate  expected_revenue_at_risk  
 0              88.28                  78890.91  
 1              62.81                 717017.06  
 2               7.45                 957249.71  
 3              35.39                1111328.00  )

In [24]:
top_50 = pred.sort_values("churn_probability", ascending=False).head(50)
top_50_path = DASHBOARD_DIR / "top_50_high_risk_customers.csv"
top_50.to_csv(top_50_path, index=False)

top_50_path

WindowsPath('C:/Users/shrut/OneDrive/Desktop/customer-churn-platform/data/processed/dashboard_tables/top_50_high_risk_customers.csv')

In [25]:
model_compare_path = REPORTS_DIR / "05_model_compare.csv"
model_compare.to_csv(model_compare_path, index=False)

logreg_top_pos_path = REPORTS_DIR / "05_logreg_top_positive.csv"
logreg_top_neg_path = REPORTS_DIR / "05_logreg_top_negative.csv"
top_pos.to_csv(logreg_top_pos_path, index=False)
top_neg.to_csv(logreg_top_neg_path, index=False)

perm_imp_path = REPORTS_DIR / "05_perm_importance.csv"
imp.to_csv(perm_imp_path, index=False)

model_compare_path, logreg_top_pos_path, logreg_top_neg_path, perm_imp_path

(WindowsPath('C:/Users/shrut/OneDrive/Desktop/customer-churn-platform/reports/05_model_compare.csv'),
 WindowsPath('C:/Users/shrut/OneDrive/Desktop/customer-churn-platform/reports/05_logreg_top_positive.csv'),
 WindowsPath('C:/Users/shrut/OneDrive/Desktop/customer-churn-platform/reports/05_logreg_top_negative.csv'),
 WindowsPath('C:/Users/shrut/OneDrive/Desktop/customer-churn-platform/reports/05_perm_importance.csv'))